In [1]:
import os
from dotenv import load_dotenv

from google.oauth2 import service_account
from google.cloud import storage

import requests

import json


In [2]:
# get environment variables
load_dotenv()

# GCS related:
service_account_key = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCS_BUCKET")

In [3]:
credentials = service_account.Credentials.from_service_account_file(
    service_account_key)

In [5]:
params = {
    "format": "geojson",
    "starttime": "2016-01-01",
    "endtime": "2026-10-01",
    "minmagnitude": 4.0,

    # Approximate California bounding box
    "minlatitude": 32.5,
    "maxlatitude": 42.0,
    "minlongitude": -124.5,
    "maxlongitude": -114.1,
}

response = requests.get("https://earthquake.usgs.gov/fdsnws/event/1/query", params=params)
data = response.json()

In [13]:
json.dumps(data)

'{"type": "FeatureCollection", "metadata": {"generated": 1790893574000, "url": "https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2016-01-01&endtime=2026-10-01&minmagnitude=4.0&minlatitude=32.5&maxlatitude=42.0&minlongitude=-124.5&maxlongitude=-114.1", "title": "USGS Earthquakes", "status": 200, "api": "2.7.0", "count": 482}, "features": [{"type": "Feature", "properties": {"mag": 4.1, "place": "20 km W of Preston, Nevada", "time": 1790499247495, "updated": 1790845459982, "tz": null, "url": "https://earthquake.usgs.gov/earthquakes/eventpage/nn00924954", "detail": "https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nn00924954&format=geojson", "felt": 1, "cdi": 1, "mmi": 3.93, "alert": "green", "status": "reviewed", "tsunami": 0, "sig": 259, "net": "nn", "code": "00924954", "ids": ",nn00924954,us6000txxy,", "sources": ",nn,us,", "types": ",dyfi,event-sequence,losspager,moment-tensor,oaf,origin,phase-data,shakemap,", "nst": 6, "dmin": 0.383, "rms": 0.195, "gap

In [17]:
# set up GCP bucket
client = storage.Client(project=project_id, credentials=credentials)
bucket = client.bucket(bucket_name)
file = bucket.blob("test_data.json")

# write to bucket
try:
    file.upload_from_string(json.dumps(data))
except Exception as e:
    print(e)

In [19]:
file = bucket.blob("earthquake_data")
file.download_as_bytes()

b'{"type": "FeatureCollection", "metadata": {"generated": 1790894574000, "url": "https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2016-01-01&endtime=2026-10-01&minmagnitude=4.0&minlatitude=32.5&maxlatitude=42.0&minlongitude=-124.5&maxlongitude=-114.1", "title": "USGS Earthquakes", "status": 200, "api": "2.7.0", "count": 482}, "features": [{"type": "Feature", "properties": {"mag": 4.1, "place": "20 km W of Preston, Nevada", "time": 1790499247495, "updated": 1790845459982, "tz": null, "url": "https://earthquake.usgs.gov/earthquakes/eventpage/nn00924954", "detail": "https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nn00924954&format=geojson", "felt": 1, "cdi": 1, "mmi": 3.93, "alert": "green", "status": "reviewed", "tsunami": 0, "sig": 259, "net": "nn", "code": "00924954", "ids": ",nn00924954,us6000txxy,", "sources": ",nn,us,", "types": ",dyfi,event-sequence,losspager,moment-tensor,oaf,origin,phase-data,shakemap,", "nst": 6, "dmin": 0.383, "rms": 0.195, "ga